# 05 — In-Sample and Out-of-Sample Event Backtests

This notebook re-evaluates the in-sample winners from notebook 03 and the two complementary out-of-sample regimes from notebook 04 using the **event** engine — a tick-for-tick replay of the strategy logic that mirrors live trading much more faithfully than the vectorised sweep used in earlier notebooks.

The three studies replayed here:

- **`in_sample_param_sweep`** — the original in-sample window (2022-01 → 2025-12) on the in-sample basket (BTC/ETH/ADA/SOL/DOT on BITVAVO/EUR). Confirms that the vector engine's ranking still holds when orders are routed bar-by-bar with realistic fills, fees and position sizing.
- **`out_sample_time_oos`** (Type A) — long-history subset (BTC/ETH on BITVAVO/EUR) over the *earlier* 2019-01 → 2021-12 regime, testing temporal robustness.
- **`out_sample_universe_oos`** (Type B) — disjoint mid-cap basket (LINK/AVAX/ATOM/ALGO/XRP on BITVAVO/EUR) over the in-sample window, testing symbol robustness.

Why event after vector? The vector engine is fast and great for screening thousands of param combinations, but it abstracts away intra-bar order routing, partial fills, slippage timing and capital availability. The event engine catches issues the vector pass glosses over, so a strategy that holds up under *both* engines is materially more credible than one that only looks good in vector.

Each event run lands on the **same `<algorithm_id>.iafbt` envelope** as its vector counterpart, populating the per-engine `event_*` slots alongside the existing `vector_*` slots. Notebook 06 can then join in-sample and OOS metrics across both engines on a stable per-strategy id.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from strategies.supertrend_ema_confirmation.strategy import (
    SupertrendEmaConfirmationStrategy as Strategy,
)

## Constants

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from investing_algorithm_framework import BacktestDateRange, Universe

data_storage_path = Path.cwd().parent / "data"
backtest_results_dir = Path.cwd().parent / "backtest_results"
top_selection_path = backtest_results_dir / "top_selection"
reports_dir = Path.cwd().parent / "reports"
figures_dir = reports_dir / "figures"

MARKET = "BITVAVO"
time_frames = ["2h", "4h", "1d"]

# ---------------------------------------------------------------------
# Three studies replayed by the event engine
# ---------------------------------------------------------------------
# This notebook re-runs the same three studies that notebooks 03 and
# 04 produced under the vector engine, but now under the event engine
# (tick-for-tick replay). The three studies, the universes they target
# and the date windows they cover are reproduced verbatim below so the
# event runs land on the same ``<algorithm_id>.iafbt`` envelopes and
# populate the per-engine ``event_*`` slots alongside the existing
# ``vector_*`` slots.
#
#  * In-sample (study ``in_sample_param_sweep``): the in-sample basket
#    over the 2022-01 → 2025-12 window — the original param sweep.
#
#  * Type A — Time OOS (study ``out_sample_time_oos``): long-history
#    subset (BTC/ETH) over the *earlier* 2019-01 → 2021-12 regime.
#    Restricted to BTC/ETH because ADA/SOL/DOT lack BITVAVO history
#    before 2021.
#
#  * Type B — Universe OOS (study ``out_sample_universe_oos``):
#    disjoint mid-cap basket over the in-sample window. Isolates
#    whether the edge is intrinsic to the strategy or merely tuned to
#    the in-sample basket.

# In-sample window + universe (must match notebook 03).
in_sample_window_date_range = BacktestDateRange(
    start_date=datetime(2022, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2025, 12, 30, tzinfo=timezone.utc),
)
in_sample_universe = Universe(
    key="in_sample_basket",
    symbols=["BTC", "ETH", "ADA", "SOL", "DOT"],
    trading_symbol="EUR",
    market=MARKET,
)

# Type A — Time OOS: strictly *before* the in-sample range so there is
# no temporal overlap and no risk of leakage from the param sweep.
out_sample_window_date_range = BacktestDateRange(
    start_date=datetime(2019, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2021, 12, 30, tzinfo=timezone.utc),
)

# Type A's basket is a *subset* of the in-sample basket: only the
# two names with full BITVAVO history back to 2019 (BTC, ETH). The
# runner's universe-subset validation will accept it because
# ``in_sample_basket ⊇ time_oos_basket``.
time_oos_universe = Universe(
    key="time_oos_basket",
    symbols=["BTC", "ETH"],
    trading_symbol="EUR",
    market=MARKET,
)

# Type B — Universe OOS: disjoint basket of mid-cap names that did
# *not* feature in the in-sample sweep. ``ALGO`` replaces the earlier
# ``MATIC``/``POL`` slot: Polygon rebranded MATIC → POL in Sept 2024,
# so neither ticker has continuous BITVAVO history over the full
# 2022-2025 window. ALGO has been listed on BITVAVO since well before
# 2022, so its parquet covers the whole range.
out_sample_universe = Universe(
    key="out_sample_basket",
    symbols=["LINK", "AVAX", "ATOM", "ALGO", "XRP"],
    trading_symbol="EUR",
    market=MARKET,
)

# ── Study identifiers ─────────────────────────────────────────────
# One study per regime. Names and descriptions are kept identical to
# notebooks 03 and 04 so the event runs collide on the same
# ``<algorithm_id>.iafbt`` envelopes and notebook 06 can join
# in-sample / OOS metrics across both engines on a stable id.
STUDY_NAME_IS = "in_sample_param_sweep"
STUDY_DESCRIPTION_IS = (
    "In-sample parameter sweep over the in_sample_basket universe "
    "(BTC/ETH/ADA/SOL/DOT on BITVAVO/EUR, 2022-01 → 2025-12)."
)

STUDY_NAME_TIME_OOS = "out_sample_time_oos"
STUDY_DESCRIPTION_TIME_OOS = (
    "Type-A out-of-sample validation: long-history subset "
    "(BTC/ETH on BITVAVO/EUR) re-evaluated over the earlier "
    "2019-01 → 2021-12 regime to test temporal robustness."
)

STUDY_NAME_UNIVERSE_OOS = "out_sample_universe_oos"
STUDY_DESCRIPTION_UNIVERSE_OOS = (
    "Type-B out-of-sample validation: disjoint mid-cap basket "
    "(LINK/AVAX/ATOM/ALGO/XRP on BITVAVO/EUR) evaluated over the "
    "in-sample 2022-01 → 2025-12 window to test symbol robustness."
)

In [ ]:
from investing_algorithm_framework import generate_rolling_backtest_windows

# Window geometry — kept identical across all three studies so any
# difference in performance is attributable to the study axis (date
# range and/or symbol basket), not to the way windows are sliced.
# Must also match notebooks 03 and 04 so the event runs land on the
# same per-window slots as their vector counterparts.
ROLLING_WINDOW_KW = dict(
    train_days=365,
    test_days=180,
    gap_days=30,
    step_days=90,
)

# In-sample: rolling windows over the in-sample date range, evaluated
# on the in-sample basket. Replays the original notebook 03 sweep
# under the event engine.
in_sample_rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=in_sample_window_date_range.start_date,
    end_date=in_sample_window_date_range.end_date,
    **ROLLING_WINDOW_KW,
)

# Type A — Time OOS: rolling windows over the *earlier* date range,
# evaluated on the long-history BTC/ETH subset.
time_oos_rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=out_sample_window_date_range.start_date,
    end_date=out_sample_window_date_range.end_date,
    **ROLLING_WINDOW_KW,
)

# Type B — Universe OOS: rolling windows over the *in-sample* date
# range, evaluated on the disjoint ``out_sample_universe`` basket.
# Reusing the in-sample window is the whole point of this regime —
# it isolates the symbol axis.
universe_oos_rolling_backtest_windows = generate_rolling_backtest_windows(
    start_date=in_sample_window_date_range.start_date,
    end_date=in_sample_window_date_range.end_date,
    **ROLLING_WINDOW_KW,
)

print(
    f"In-sample:             {len(in_sample_rolling_backtest_windows)} windows "
    f"over {in_sample_window_date_range.start_date.date()} → "
    f"{in_sample_window_date_range.end_date.date()}"
)
print(
    f"Type A (time OOS):     {len(time_oos_rolling_backtest_windows)} windows "
    f"over {out_sample_window_date_range.start_date.date()} → "
    f"{out_sample_window_date_range.end_date.date()}"
)
print(
    f"Type B (universe OOS): {len(universe_oos_rolling_backtest_windows)} windows "
    f"over {in_sample_window_date_range.start_date.date()} → "
    f"{in_sample_window_date_range.end_date.date()}"
)

Only select the best strategy because off time constraints. In a real research process, you'd probably want to event-test the top 5-10 vector survivors to get a better sense of the robustness of the signal across different param combinations.

In [ ]:
from investing_algorithm_framework import (
    BacktestEvaluationFocus, build_index, rank_index, LocalDirStore,
)

# 1. (Re)build the Tier-1 index over the archived top selection so we
#    can rank without decoding every bundle.
build_index(
    str(top_selection_path),
    show_progress=True,
    incremental=True,
)

# 2. Re-rank with the same BALANCED focus used in the in-sample sweep.
top = rank_index(
    str(top_selection_path),
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study="in_sample_param_sweep",
)

# 3. Materialise the winners (deduplicate by bundle_path so dual-engine
#    bundles aren't opened twice) and pull the *original* in-sample
#    grid variant out of ``metadata["params"]``. We use the metadata
#    payload — not ``bt.parameters`` — because notebook 03 hashed the
#    variant (minus ``_grid_profile``) to derive the in-sample
#    ``algorithm_id``. Hashing the same shape here is what makes the
#    OOS bundle collide with the in-sample envelope.
store = LocalDirStore(str(top_selection_path))
seen_bundles = set()
top_backtests = []
for row in top:
    bp = row["bundle_path"]
    if bp in seen_bundles:
        continue
    seen_bundles.add(bp)
    top_backtests.append(store.open(bp))

top_param_variations = []
skipped = []
for bt in top_backtests:
    md = bt.get_metadata() or {}
    variant = dict(md.get("params") or {})
    if not variant:
        # Legacy bundles without metadata["params"] fall back to the
        # canonical ``Backtest.parameters`` slot — the framework auto-
        # derived id will still match because both notebooks now use
        # the same hashing rule.
        variant = dict(bt.parameters or {})
    if not variant:
        skipped.append(bt.algorithm_id)
        continue
    top_param_variations.append(variant)

print(
    f"Loaded {len(top_param_variations)} param sets "
    f"from {len(top_backtests)} top-selection bundles"
)
if skipped:
    print(f"Skipped {len(skipped)} bundle(s) with no saved params: {skipped}")


In [ ]:
from investing_algorithm_framework import (
    create_backtest_metrics_table, create_trade_metrics_table,
)
from tests.domain import backtests

number_one = top_backtests[0]

# NOTE: the metric-table helpers operate on materialised ``Backtest``
# objects (the ``backtests`` list opened via ``store.open`` in the
# previous cell). ``top`` from ``rank_index`` is a list of raw SQLite
# index rows (dicts) and is only used for ranking / pruning.
#
# We pass ``study=STUDY_NAME`` so the table helpers resolve the
# correct vector / event slot in multi-study bundles via
# ``Backtest.get_summary(engine, study=...)`` and
# ``Backtest.get_runs(engine, study=...)``. Without it, multi-study
# bundles would raise under the framework's default-study rule.

print(
    "Summary backtest metrics for top strategies "
    "(vector engine, sorted by Sharpe ratio):\n"
)
print(create_backtest_metrics_table(
    [number_one],
    engine="vector",
    study=STUDY_NAME_IS,
    sort_by="sharpe_ratio",
))

print(
    "Backtest metrics by backtest window for top strategies "
    "(vector engine):\n"
)
print(create_backtest_metrics_table(
    [number_one],
    engine="vector",
    study=STUDY_NAME_IS,
    level="run",
    window=[w["train_range"] for w in in_sample_rolling_backtest_windows],
))

print(
    "Summary trade metrics for top strategies "
    "(vector engine, sorted by profit factor):\n"
)
print(create_trade_metrics_table(
    [number_one],
    engine="vector",
    study=STUDY_NAME_IS,
    sort_by="profit_factor",
))

print(
    "Trade metrics by backtest window for top strategies "
    "(vector engine):\n"
)
print(create_trade_metrics_table(
    [number_one],
    engine="vector",
    study=STUDY_NAME_IS,
    level="run",
    window=[w["train_range"] for w in in_sample_rolling_backtest_windows],
))


In [ ]:
from investing_algorithm_framework import (
    BacktestEvaluationFocus, build_index, rank_index, LocalDirStore,
)

# 1. (Re)build the Tier-1 index over the archived top selection so we
#    can rank without decoding every bundle.
build_index(
    str(top_selection_path),
    show_progress=True,
    incremental=True,
)

# 2. Re-rank with the same BALANCED focus used in the in-sample sweep.
top = rank_index(
    str(top_selection_path),
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study="in_sample_param_sweep",
)

# 3. Materialise the winners (deduplicate by bundle_path so dual-engine
#    bundles aren't opened twice) and pull the *original* in-sample
#    grid variant out of ``metadata["params"]``. We use the metadata
#    payload — not ``bt.parameters`` — because notebook 03 hashed the
#    variant (minus ``_grid_profile``) to derive the in-sample
#    ``algorithm_id``. Hashing the same shape here is what makes the
#    OOS bundle collide with the in-sample envelope.
store = LocalDirStore(str(top_selection_path))
seen_bundles = set()
top_backtests = []
for row in top:
    bp = row["bundle_path"]
    if bp in seen_bundles:
        continue
    seen_bundles.add(bp)
    top_backtests.append(store.open(bp))

top_param_variations = []
skipped = []
for bt in top_backtests:
    md = bt.get_metadata() or {}
    variant = dict(md.get("params") or {})
    if not variant:
        # Legacy bundles without metadata["params"] fall back to the
        # canonical ``Backtest.parameters`` slot — the framework auto-
        # derived id will still match because both notebooks now use
        # the same hashing rule.
        variant = dict(bt.parameters or {})
    if not variant:
        skipped.append(bt.algorithm_id)
        continue
    top_param_variations.append(variant)

print(
    f"Loaded {len(top_param_variations)} param sets "
    f"from {len(top_backtests)} top-selection bundles"
)
if skipped:
    print(f"Skipped {len(skipped)} bundle(s) with no saved params: {skipped}")


## Strategy Initialization

In [ ]:
from investing_algorithm_framework import generate_algorithm_id
from investing_algorithm_framework.domain import tqdm


def initialize_strategies(
    strategy_class,
    param_variations,
    symbols,
    market,
    trading_symbol="EUR",
    filter_fn=None,
):
    """Re-instantiate the in-sample winners for an out-of-sample run.

    Each ``variant`` is the in-sample grid variant we recovered from
    ``bt.metadata["params"]`` in the loader cell. We strip the
    underscore-prefixed metadata (``_grid_profile`` etc.) and hash the
    same stable subset notebook 03 used to derive the in-sample
    ``algorithm_id``. Passing that id explicitly here makes each OOS
    bundle land in the same ``<algorithm_id>.iafbt`` envelope as its
    in-sample winner (multi-study slot).
    """
    strategies = []

    for variant in tqdm(
        param_variations, desc="Initializing strategies", colour="green"
    ):
        # Mirror notebook 03: drop underscore-prefixed metadata before
        # hashing and before passing to the strategy constructor.
        strategy_params = {
            k: v for k, v in variant.items() if not k.startswith("_")
        }

        strategy = strategy_class(
            algorithm_id=generate_algorithm_id(params=strategy_params),
            symbols=symbols,
            trading_symbol=trading_symbol,
            market=market,
            metadata={
                "params": variant,
                "symbols": symbols,
                "market": market,
            },
            **strategy_params,
        )
        strategies.append(strategy)

    if filter_fn:
        strategies = [s for s in strategies if filter_fn(s)]

    return strategies


## Run event backtest on In-Sample Study

In [ ]:
import os

from investing_algorithm_framework import (
    create_app, RESOURCE_DIRECTORY, PortfolioConfiguration, DATA_DIRECTORY
)

# Engine used for the OOS validation. v9.0 ``Backtest`` carries
# separate ``vector_summary`` and ``event_summary`` slots (with their
# own ``*_runs``) so any per-bundle summary access MUST go through
# ``backtest.get_summary(engine)`` — the legacy single
# ``backtest.backtest_summary`` attribute no longer exists.
ENGINE = "event"

# ── Type A — Time OOS ──────────────────────────────────────────────
# Long-history subset of the in-sample basket (BTC, ETH), *earlier*
# date window. Tests whether the surviving params generalise to a
# different market regime (pre-2022 covers the 2019-2021 cycle
# including the 2020 crash and the late-2021 peak — none of which the
# in-sample sweep saw). ADA, SOL and DOT are excluded here because
# their BITVAVO history doesn't reach far enough back.
backtest_windows_time_oos = [
    w["train_range"] for w in time_oos_rolling_backtest_windows
]

strategies_time_oos = initialize_strategies(
    strategy_class=Strategy,
    param_variations=top_param_variations,
    symbols=list(time_oos_universe.symbols),
    market=time_oos_universe.market,
    trading_symbol=time_oos_universe.trading_symbol,
)

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})
app.add_portfolio_configuration(
    PortfolioConfiguration(
        initial_balance=1000,
        market=time_oos_universe.market,
        trading_symbol=time_oos_universe.trading_symbol,
    )
)

# Phase 2b: passing ``universes=[time_oos_universe]`` stamps the
# narrower BTC/ETH descriptor on every Type-A bundle. Notebook 05 can
# join in-sample vs. OOS results on a stable ``universe_key`` even
# though the symbol set is a strict subset of the in-sample basket.
backtests_time_oos = app.run_event_backtests(
    backtest_date_ranges=backtest_windows_time_oos,
    strategies=strategies_time_oos,
    universes=[time_oos_universe],
    risk_free_rate=0.027,
    continue_on_error=False,
    use_checkpoints=True,
    backtest_storage_directory=top_selection_path,
    show_progress=True,
    n_workers=os.cpu_count() - 3,
    dynamic_position_sizing=True,
    study_name=STUDY_NAME_TIME_OOS,
    study_description=STUDY_DESCRIPTION_TIME_OOS,
)

print(
    f"\nType A (time OOS) complete — {len(backtests_time_oos)} backtests "
    f"after filtering"
)
